# 03 — Customer RFM and Cohort Analysis

This notebook explores customer value and retention patterns using the SQL outputs generated in Notebook 02. RFM means **Recency, Frequency and Monetary value**. Cohort analysis tracks the share of an acquisition-month group that returns in later months.

These are descriptive customer analytics; they are separate from the supervised repeat-purchase model.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "reports" / "sql").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
SQL_OUTPUT = PROJECT_ROOT / "reports" / "sql"
if not SQL_OUTPUT.exists():
    raise FileNotFoundError("Run Notebook 02 first to generate reports/sql/*.csv")

rfm = pd.read_csv(SQL_OUTPUT / "rfm_segments.csv")
cohorts = pd.read_csv(SQL_OUTPUT / "cohort_retention.csv")
concentration = pd.read_csv(SQL_OUTPUT / "revenue_concentration.csv")
display(rfm)
display(cohorts.head())
display(concentration.head())

## 1. RFM segment mix

In [ ]:
ax = rfm.set_index("segment")["customers"].sort_values().plot(
    kind="barh", title="Customers by RFM segment", xlabel="Customers"
)
ax.figure.tight_layout()

## 2. Cohort retention matrix

Convert the long cohort table into a matrix. Newer cohorts naturally have fewer observed months; blank cells are not zero retention.

In [ ]:
retention_matrix = cohorts.pivot(
    index="cohort_month", columns="month_number", values="retention_rate"
)
display(retention_matrix.style.format("{:.1%}", na_rep="—"))

## 3. Revenue concentration

The SQL output ranks identified customers by cumulative revenue share. This helps quantify reliance on high-value customers.

In [ ]:
top_n = min(20, len(concentration))
display(concentration.head(top_n))
if len(concentration):
    print("Revenue share contributed by top 10 customers:",
          f"{concentration.head(10)['revenue_gbp'].sum() / concentration['revenue_gbp'].sum():.1%}")

## Business interpretation

Write a short memo with:
1. The largest RFM segments and what they imply.
2. Whether cohort retention appears to decay quickly or gradually.
3. The degree of customer revenue concentration.
4. One practical retention hypothesis to test—not a causal claim.